# AutoClips on Google Colab (GPU)

Runs the full API + Telegram bot with **CUDA Whisper** (5-10x faster transcription than CPU).

Steps once per browser:
1. **Runtime ▸ Change runtime type ▸ T4 GPU** (top-right).
2. Run cells top-to-bottom.
3. Keep this tab open while working — Colab kills the session when idle (~90 min) or after ~12h.

Stores `storage/` (downloads, transcripts, clips) on Google Drive so nothing is lost between sessions.

## 1. Mount Google Drive (persists downloads/clips/transcripts)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
print("Drive mounted ✓")

## 2. Upload the project code

Choose **one**:
- **(a)** Zip the `autoclips-python` folder locally and upload it via the file browser on the left, or
- **(b)** `git clone` a remote copy (uncomment the cell).

In [ ]:
import os, shutil, zipfile

WORK = "/content/autoclips-python"
os.makedirs(WORK, exist_ok=True)

# (b) unchanged? uncomment and use the subprocess cell below instead.
print("Use the file browser to upload your zip inside /content/, then run the next cell.")

In [ ]:
import glob, os, shutil, zipfile

zips = sorted(glob.glob("/content/*.zip"))
if not zips:
    raise SystemExit("No .zip found in /content/. Upload your autoclips-python.zip first.")
z = zips[-1]
print("Extracting", z)
with zipfile.ZipFile(z) as zf:
    zf.extractall("/content/")
os.remove(z)

# If the zip has one top-level folder that isn't "autoclips-python", rename it.
folders = [d for d in os.listdir("/content")
           if os.path.isdir(f"/content/{d}") and not d.startswith(".")]
if "autoclips-python" not in folders and len(folders) == 1:
    os.rename(f"/content/{folders[0]}", WORK)
print("Project ready at", WORK)
print(sorted(os.listdir(WORK))[:15])

## 3. Install ffmpeg + Python deps

In [ ]:
!apt-get -qq update && apt-get -qq install -y ffmpeg fonts-dejavu > /dev/null
print("ffmpeg:", !ffmpeg -version | head -1)

In [ ]:
%cd /content/autoclips-python
!pip -q install -r requirements.txt
print("deps installed")

## 4. Verify the GPU is visible to faster-whisper

In [ ]:
import ctranslate2
print("CUDA devices visible to CTranslate2:", ctranslate2.get_cuda_device_count())
if ctranslate2.get_cuda_device_count() == 0:
    raise SystemExit("No GPU! Go to Runtime > Change runtime type > T4 GPU, then restart.")

## 5. Configure `.env`

Fill in the required values (Telegram + Gemini). Kept in Drive so you don't retype each session.

In [ ]:
import os
os.chdir("/content/drive/MyDrive/autoclips-config")

# ── Edit these values ─────────────────────────────────────────────
TELEGRAM_BOT_TOKEN = "123456:ABC-DEF..."
TELEGRAM_CHAT_ID   = "your-chat-id"
GEMINI_API_KEY     = "your-gemini-key"
WHISPER_MODEL_SIZE = "large-v3-turbo"  # small | medium | large-v3-turbo (T4 sweet spot)
FFMPEG_PRESET      = "ultrafast"      # veryfast | ultrafast
FACE_MODEL         = "yolo"           # yolo (GPU) | yunet (no dep) | res10 (legacy)
# ──────────────────────────────────────────────────────────────────

os.makedirs("autoclips-config", exist_ok=True)
os.chdir("/content/autoclips-python")

if os.path.exists(".env"):
    print("Keeping existing .env (delete it to regenerate)")
else:
    env = f"""TELEGRAM_BOT_TOKEN={TELEGRAM_BOT_TOKEN}
TELEGRAM_CHAT_ID={TELEGRAM_CHAT_ID}
GEMINI_API_KEY={GEMINI_API_KEY}
WHISPER_MODEL_SIZE={WHISPER_MODEL_SIZE}
WHISPER_DEVICE=cuda
WHISPER_COMPUTE_TYPE=float16
WHISPER_VAD_FILTER=true
FFMPEG_PRESET={FFMPEG_PRESET}
FACE_MODEL={FACE_MODEL}
FACE_SAMPLE_FPS=1.0
SMOOTH_CROP=true
SNAP_TO_SILENCE=true
LOG_LEVEL=INFO
"""
    open(".env", "w").write(env)
    print("Wrote .env with CUDA settings ✓")

## 6. Keep storage/ on Drive (survives restarts)

Downloads, transcripts, clips and logs persist under your Drive. First run moves the folder; later runs reuse it.

In [ ]:
import os, shutil
os.chdir("/content/autoclips-python")

STORE = "/content/drive/MyDrive/autoclips-storage"
if not os.path.exists(STORE):
    os.makedirs(STORE, exist_ok=True)

# Replace the ephemeral storage/ with a symlink to Drive
for d in ["downloads", "clips", "tmp", "videos", "logs"]:
    os.makedirs(f"{STORE}/{d}", exist_ok=True)
shutil.rmtree("storage", ignore_errors=True)
os.symlink(STORE, "storage")
print("storage/ ->", os.readlink("storage"))

## 7. Start the server + Telegram bot (background)

Runs uvicorn (which also starts the bot) in the background. Watch `storage/logs/app.log` in cells 8-9.

In [ ]:
import os, subprocess, time
os.chdir("/content/autoclips-python")

# Kill any previous instance from an old session of this same run
subprocess.run(["pkill", "-f", "uvicorn app.main:app"], capture_output=True)
time.sleep(2)

log = open("storage/logs/colab_server.log", "a", buffering=1)
subprocess.Popen(
    ["/usr/local/bin/uvicorn", "app.main:app", "--port", "8000"],
    stdout=log, stderr=subprocess.STDOUT,
)
print("Server starting... wait ~30-60s for the bot to come online.")
print("Then just message your bot a YouTube link from Telegram.")

## 8. Watch logs (re-run this cell anytime)

In [ ]:
!tail -n 40 storage/logs/app.log

## 9. Keep the session alive while you work

Run once — a background thread polls the server every 30s so the session doesn't go idle.
Still subject to Colab's ~12h hard cap; restart when it ends.

In [ ]:
import threading, time, urllib.request

def keep_alive():
    while True:
        try:
            urllib.request.urlopen("http://localhost:8000/health", timeout=5)
        except Exception:
            pass
        time.sleep(30)

t = threading.Thread(target=keep_alive, daemon=True)
t.start()
print("Keep-alive running. When done, re-run cell 7 next session to restart the server.")

## Notes / troubleshooting
- **Transcription is now GPU — check it:** `WHISPER_DEVICE=cuda`, `WHISPER_COMPUTE_TYPE=float16`, `WHISPER_VAD_FILTER=true` in `.env`. The first transcribe also downloads the model (~1.6GB for `large-v3-turbo`) so it's slower once.
- **Face on T4:** `FACE_MODEL=yolo` uses the T4 GPU at 1fps; on CPU boxes set `FACE_MODEL=yunet` (no new dep) or `res10`.
- **Session lost?** Drive holds `.env`, code zip (re-upload) and `storage/`. Re-run cells 5-7 only.
- **Rate limits:** Telegram bot uses long polling — no public URL needed.
- **YouTube uploads (Approve buttons)** need `YT_*` vars added to `.env` — same as local.
- **Out of VRAM:** switch to `WHISPER_MODEL_SIZE=medium`, and transcribe before face detection (sequential) so both don't sit in VRAM at once.